# Bronze Ingest: USGS Earthquakes

Downloads the USGS "all earthquakes, past month" GeoJSON feed and persists
it as a managed Spark table, flattened from GeoJSON `Feature` objects into
flat rows. This is the `bronze` layer.

This feed updates continuously (every few minutes), so re-running this
notebook always pulls fresh data — good for demos that want data that
visibly changes between runs. See
https://earthquake.usgs.gov/earthquakes/feed/v1.0/geojson.php for feed
docs and other magnitude/time-window options.

Re-runnable: uses `CREATE DATABASE IF NOT EXISTS` and `mode("overwrite")`,
so re-running just refreshes the table.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.getOrCreate()

## Configuration

Swap `all_month` for `all_week`/`all_day`/`all_hour` (smaller, more
"live") or prefix with a magnitude filter (e.g. `4.5_month`) — see the
feed docs linked above for the full list of available feeds.

In [ ]:
DATABASE_NAME = "earthquake_data"
TABLE_NAME = "earthquakes"
SOURCE_URL = "https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/all_month.geojson"
LOCAL_FILE = "/tmp/usgs_earthquakes_all_month.geojson"

## Download source file

In [ ]:
import urllib.request

urllib.request.urlretrieve(SOURCE_URL, LOCAL_FILE)

## Load dataset

A GeoJSON `FeatureCollection` is one JSON object holding a `features`
array — read with `multiLine`, then explode `features` into rows and pull
out the fields used for demos.

In [ ]:
raw_df = spark.read.option("multiLine", True).json(LOCAL_FILE)

df = (
    raw_df
    .select(F.explode("features").alias("f"))
    .select(
        F.col("f.id").alias("id"),
        F.col("f.properties.mag").alias("magnitude"),
        F.col("f.properties.magType").alias("mag_type"),
        F.col("f.properties.place").alias("place"),
        (F.col("f.properties.time") / 1000).cast("timestamp").alias("event_time"),
        F.col("f.properties.type").alias("event_type"),
        F.col("f.properties.status").alias("status"),
        F.col("f.properties.tsunami").alias("tsunami"),
        F.col("f.properties.sig").alias("significance"),
        F.col("f.geometry.coordinates").getItem(0).alias("longitude"),
        F.col("f.geometry.coordinates").getItem(1).alias("latitude"),
        F.col("f.geometry.coordinates").getItem(2).alias("depth_km"),
    )
)

## Create schema/database

In [ ]:
spark.sql(f"""
CREATE DATABASE IF NOT EXISTS {DATABASE_NAME}
COMMENT 'USGS earthquake feed data used for analytics demonstrations'
""")

## Persist as a managed Spark table

In [ ]:
df.write \
    .mode("overwrite") \
    .saveAsTable(f"{DATABASE_NAME}.{TABLE_NAME}")

## Verify

In [ ]:
spark.sql(f"SELECT COUNT(*) AS total_events FROM {DATABASE_NAME}.{TABLE_NAME}").show()
spark.sql(f"DESCRIBE {DATABASE_NAME}.{TABLE_NAME}").show(truncate=False)